# Deploy Object Detection Training Pipeline via KFP SDK

This notebook deploys the YOLOv5 object-detection training pipeline
to **Data Science Pipelines** on OpenShift AI using the KFP Python SDK.

It replaces the Elyra-based `model-training-cpu.pipeline` with an
equivalent KFP v2 pipeline built entirely in Python.

### Pipeline steps

| # | Step | Description |
|---|------|-------------|
| 1 | Preprocessing | Split images into train / val / test |
| 2 | Training | Train YOLOv5 model (CPU) |
| 3 | Conversion | Export model to ONNX |
| 4 | Upload | Upload ONNX model to S3 |

All pipeline step logic is **self-contained** inside each KFP
component — no scripts or code need to be copied to the PVC.
The PVC is used only for training data and intermediate model files.

### Prerequisites

* An OpenShift AI workbench
* The `workbench-bucket-ai-connection` Kubernetes secret for S3 credentials
* Data Science Pipelines (DSPA) configured in the namespace
* Training images on the `object-detection-training-pvc` PVC at
  `/data/custom_training_images/<class>/images/*.jpg`

## 1 — Install dependencies

In [ ]:
!pip install -q kfp kfp-kubernetes

## 2 — Compile the pipeline (sanity check)

Compile the pipeline to a `pipeline.yaml` IR file.
This verifies that the pipeline definition is valid before
uploading to the server.

In [ ]:
import os, sys

NOTEBOOK_DIR = os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from kfp.compiler import Compiler
from kfp_pipeline import model_training_pipeline

compiled_path = os.path.join(NOTEBOOK_DIR, 'pipeline.yaml')
Compiler().compile(model_training_pipeline, package_path=compiled_path)
print(f'✅ Pipeline compiled to {compiled_path}')

## 2b — Visualize the pipeline

Render the compiled pipeline DAG directly in the notebook.
No system packages required — uses pure HTML + SVG.

In [ ]:
import yaml
from IPython.display import HTML

with open(compiled_path) as f:
    spec = list(yaml.safe_load_all(f))[0]

tasks = spec['root']['dag']['tasks']
components = spec.get('components', {})

# Build ordered task list following dependencies
ordered = []
remaining = dict(tasks)
while remaining:
    for name, task in list(remaining.items()):
        deps = task.get('dependentTasks', [])
        if all(d in ordered for d in deps) or not deps:
            ordered.append(name)
            del remaining[name]
            break

colors = {
    'preprocess-data': '#34a853',
    'train-model':     '#4285f4',
    'convert-model':   '#fbbc04',
    'upload-model':    '#ea4335',
}

# Generate SVG
box_w, box_h, gap = 170, 70, 50
total_w = len(ordered) * (box_w + gap) - gap + 40
svg_h = 120

svg_parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{total_w}" height="{svg_h}">']
svg_parts.append('<style>text { font-family: Helvetica, Arial, sans-serif; fill: white; }</style>')

for i, name in enumerate(ordered):
    x = 20 + i * (box_w + gap)
    y = (svg_h - box_h) // 2
    fill = colors.get(name, '#888')

    comp_name = tasks[name].get('componentRef', {}).get('name', '')
    comp = components.get(comp_name, {})
    params = list(comp.get('inputDefinitions', {}).get('parameters', {}).keys())

    label = name.replace('-', ' ').title()
    param_str = ', '.join(params) if params else ''

    # Box
    svg_parts.append(
        f'<rect x="{x}" y="{y}" width="{box_w}" height="{box_h}" '
        f'rx="10" fill="{fill}" />'
    )
    # Title
    svg_parts.append(
        f'<text x="{x + box_w//2}" y="{y + 28}" text-anchor="middle" '
        f'font-size="13" font-weight="bold">{label}</text>'
    )
    # Params
    if param_str:
        svg_parts.append(
            f'<text x="{x + box_w//2}" y="{y + 48}" text-anchor="middle" '
            f'font-size="9" opacity="0.85">{param_str}</text>'
        )

    # Arrow to next
    if i < len(ordered) - 1:
        ax = x + box_w
        ay = svg_h // 2
        bx = ax + gap
        svg_parts.append(
            f'<line x1="{ax}" y1="{ay}" x2="{bx - 8}" y2="{ay}" '
            f'stroke="#888" stroke-width="2" />'
        )
        svg_parts.append(
            f'<polygon points="{bx},{ay} {bx-10},{ay-5} {bx-10},{ay+5}" fill="#888" />'
        )

svg_parts.append('</svg>')
HTML(''.join(svg_parts))

## 3 — Configure pipeline parameters

Adjust the values below for your training run.

In [ ]:
PIPELINE_NAME      = 'object-detection-training-sdk'
EXPERIMENT_NAME    = 'object-detection'

PIPELINE_PARAMS = {
    'batch_size':           16,
    'epochs':               300,
    'base_model':           'yolov5n',
    'model_object_prefix':  'model',
    'class_names':          'Fedora',     # comma-separated if multiple classes
}

## 4 — Deploy and run the pipeline

The `KfpPipeline` helper connects to the DSP API server in
the current namespace, uploads (or updates) the pipeline, and
starts a run.

> This cell must run inside an **OpenShift AI workbench**
> (it reads the service-account token to authenticate).

In [ ]:
from kfp_client import KfpPipeline
from kfp_pipeline import model_training_pipeline

pipeline = KfpPipeline(
    sdk_pipeline=model_training_pipeline,
    pipeline_name=PIPELINE_NAME,
)

pipeline.run_with_parameters(
    pipeline_parameters=PIPELINE_PARAMS,
    experiment_name=EXPERIMENT_NAME,
)

print('\n🎉 Pipeline submitted — check the Data Science Pipelines UI for progress.')

---

### Troubleshooting

| Issue | Fix |
|-------|-----|
| `FileNotFoundError: namespace` | Run this notebook inside an OpenShift AI workbench (needs a service-account token). |
| `ConnectionError: ds-pipeline-dspa` | Ensure a Data Science Pipelines Application (DSPA) is configured in your project. |
| Preprocessing fails — "No .jpg images" | Verify training images exist on the PVC at `/data/custom_training_images/<class>/images/`. |
| Training fails — "yolov5" import error | The runtime image may be missing dependencies. Update the image tag in `kfp_pipeline.py`. |
| S3 upload fails | Check that the `workbench-bucket-ai-connection` secret exists and has valid S3 credentials. |